# Peripheral Binding Smoke Tests

Safe smoke-test cells for ASI Tiger and SyncBoard bindings. Run the setup cell first, update the serial port constants, then run one peripheral cell at a time.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

from contextlib import suppress
from pathlib import Path
import os
import sys
import time
from typing import Any


os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)


def find_workspace_root() -> Path:
    """
    Return the workspace root that contains the evomachine repository.

    Parameters
    ----------
    None

    Returns
    -------
    Path
        Workspace root used to add local sibling repositories to sys.path.
    """
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "evomachine" / "evomachine").is_dir():
            return candidate
    return current


def add_import_path(path: Path) -> None:
    """
    Add an existing package root to sys.path if it is not already present.

    Parameters
    ----------
    path
        Candidate package root to add for notebook imports.

    Returns
    -------
    None
    """
    if path.exists():
        path_text = str(path.resolve())
        if path_text not in sys.path:
            sys.path.insert(0, path_text)


def list_serial_ports() -> list[dict[str, str]]:
    """
    Return available serial ports for choosing hardware constants.

    Parameters
    ----------
    None

    Returns
    -------
    list[dict[str, str]]
        Serial port metadata dictionaries, or one error dictionary if pyserial is unavailable.
    """
    try:
        from serial.tools import list_ports
    except Exception as error:
        return [{"error": f"Could not import pyserial list_ports: {error}"}]
    return [
        {
            "device": port.device,
            "description": port.description,
            "hwid": port.hwid,
        }
        for port in list_ports.comports()
    ]


def require_object(name: str) -> Any:
    """
    Return a global object by name or raise a clear notebook-order error.

    Parameters
    ----------
    name
        Global variable name expected to contain an initialised controller or peripheral.

    Returns
    -------
    Any
        The object stored under name in the notebook global namespace.
    """
    value = globals().get(name)
    if value is None:
        raise RuntimeError(f"Run the {name} setup cell before this cell.")
    return value


def remember_peripheral(name: str, peripheral: Any) -> Any:
    """
    Store a peripheral globally and add it to the cleanup list.

    Parameters
    ----------
    name
        Global variable name that should point to the peripheral.
    peripheral
        Peripheral instance created by a smoke-test cell.

    Returns
    -------
    Any
        The same peripheral instance, for inline assignment.
    """
    globals()[name] = peripheral
    if peripheral not in created_peripherals:
        created_peripherals.append(peripheral)
    return peripheral


def remember_controller(name: str, controller: Any) -> Any:
    """
    Store a controller globally and add it to the cleanup list.

    Parameters
    ----------
    name
        Global variable name that should point to the controller.
    controller
        Peripheral controller instance created by a smoke-test cell.

    Returns
    -------
    Any
        The same controller instance, for inline assignment.
    """
    globals()[name] = controller
    if controller not in created_controllers:
        created_controllers.append(controller)
    return controller


WORKSPACE_ROOT = find_workspace_root()
MAIN_REPO_ROOT = WORKSPACE_ROOT / "evomachine"
add_import_path(MAIN_REPO_ROOT)
add_import_path(WORKSPACE_ROOT / "asitiger")
add_import_path(WORKSPACE_ROOT / "sync_board")

from evomachine.bindings.asitiger.autofocus import TigerAutofocus
from evomachine.bindings.asitiger.filterwheel import TigerFilterWheel
from evomachine.bindings.asitiger.leds import TigerLedSource
from evomachine.bindings.asitiger.peripheralcontroller import TigerPeripheralController
from evomachine.bindings.asitiger.stage import TigerStage
from evomachine.bindings.syncboard.leds import SyncBoardLedSource
from evomachine.bindings.syncboard.peripheralcontroller import SyncBoardPeripheralController
from evomachine.bindings.syncboard.photodiode import SyncBoardPhotodiode
from evomachine.coordinates import Coordinate
from evomachine.peripherals.photodiode import PhotodiodeReadingRange
from evomachine.types import FilterWheelType, LEDType

TIGER_PORT = "/dev/ttyUSB0"
SYNCBOARD_PORT = "/dev/ttyACM0"

FOV_STEP_SIZE = 100.0
STAGE_SMOKE_COORDINATE = Coordinate(x=0, y=0, z=0)
RUN_STAGE_MOVE = False

TIGER_TEST_LED = LEDType.LED_OVERHEAD_TIGER
SYNCBOARD_TEST_LED = LEDType.LED_450_NM
LED_TEST_BRIGHTNESS = 1.0
LED_TEST_DURATION_MS = 100.0

RUN_FILTER_CHANGE = False
TARGET_FILTER = FilterWheelType.FILTER_527nm

RUN_CRISP_CALIBRATION = False
LOCK_AFTER_CRISP_CALIBRATION = False

PHOTODIODE_CHANNEL = 8
PHOTODIODE_READING_RANGE = PhotodiodeReadingRange(0.0, 1.0)

tiger_controller = None
tiger_stage = None
tiger_filter_wheel = None
tiger_led_source = None
tiger_autofocus = None
syncboard_controller = None
syncboard_led_source = None
syncboard_photodiode = None
created_controllers = []
created_peripherals = []

{
    "workspace_root": WORKSPACE_ROOT,
    "main_repo_root": MAIN_REPO_ROOT,
    "serial_ports": list_serial_ports(),
    "tiger_port": TIGER_PORT,
    "syncboard_port": SYNCBOARD_PORT,
}


## ASI Tiger

In [ ]:
# TigerPeripheralController

tiger_controller = remember_controller(
    "tiger_controller",
    TigerPeripheralController.from_serial_port(port=TIGER_PORT),
)
tiger_controller.initialise()

{
    "name": tiger_controller.name,
    "port": TIGER_PORT,
    "is_initialised": tiger_controller.is_initialised(),
    "is_alive": tiger_controller.is_alive(),
    "card_address_crisp": tiger_controller.card_address_crisp,
    "card_address_led": tiger_controller.card_address_led,
    "card_address_filter_wheel": tiger_controller.card_address_filter_wheel,
}


In [ ]:
# TigerStage

tiger_controller = require_object("tiger_controller")
tiger_stage = remember_peripheral(
    "tiger_stage",
    TigerStage(peripheral_ctrl=tiger_controller, fov_step_size=FOV_STEP_SIZE),
)
tiger_stage.initialise()
initial_coordinate = tiger_stage.get_coordinates()
limits = tiger_stage.get_stage_limits()

if RUN_STAGE_MOVE:
    tiger_stage.move(target=STAGE_SMOKE_COORDINATE, block=True)

{
    "name": tiger_stage.name,
    "is_initialised": tiger_stage.is_initialised(),
    "is_alive": tiger_stage.is_alive(),
    "initial_coordinate": initial_coordinate,
    "current_coordinate": tiger_stage.get_coordinates(),
    "stage_limits": limits,
    "ran_stage_move": RUN_STAGE_MOVE,
}


In [ ]:
# TigerFilterWheel

tiger_controller = require_object("tiger_controller")
available_filters = list(TigerFilterWheel.DEFAULT_FILTER_WHEEL_SETTINGS)
tiger_filter_wheel = remember_peripheral(
    "tiger_filter_wheel",
    TigerFilterWheel(
        peripheral_ctrl=tiger_controller,
        available_filters=available_filters,
    ),
)
tiger_filter_wheel.initialise()
initial_filter = tiger_filter_wheel.get_filter_wheel()

if RUN_FILTER_CHANGE:
    tiger_filter_wheel.set_filter_wheel(filter_type=TARGET_FILTER, force=True)

{
    "name": tiger_filter_wheel.name,
    "is_initialised": tiger_filter_wheel.is_initialised(),
    "is_alive": tiger_filter_wheel.is_alive(),
    "available_filters": tiger_filter_wheel.get_available_filters(),
    "initial_filter": initial_filter,
    "current_filter": tiger_filter_wheel.get_filter_wheel(),
    "ran_filter_change": RUN_FILTER_CHANGE,
}


In [ ]:
# TigerLedSource

tiger_controller = require_object("tiger_controller")
tiger_led_source = remember_peripheral(
    "tiger_led_source",
    TigerLedSource(
        peripheral_ctrl=tiger_controller,
        available_leds=[TIGER_TEST_LED],
    ),
)
tiger_led_source.initialise()
try:
    tiger_led_source.set_led(
        led_type=TIGER_TEST_LED,
        brightness=LED_TEST_BRIGHTNESS,
        duration=LED_TEST_DURATION_MS,
    )
    time.sleep(LED_TEST_DURATION_MS / 1000.0 + 0.05)
finally:
    tiger_led_source.disable_led()

{
    "name": tiger_led_source.name,
    "is_initialised": tiger_led_source.is_initialised(),
    "is_alive": tiger_led_source.is_alive(),
    "available_leds": tiger_led_source.get_available_leds(),
    "test_led": TIGER_TEST_LED,
    "brightness": LED_TEST_BRIGHTNESS,
    "duration_ms": LED_TEST_DURATION_MS,
    "state_after_disable": tiger_led_source.get_led_state(TIGER_TEST_LED),
}


In [ ]:
# TigerAutofocus

tiger_controller = require_object("tiger_controller")
tiger_autofocus = remember_peripheral(
    "tiger_autofocus",
    TigerAutofocus(peripheral_ctrl=tiger_controller),
)
tiger_autofocus.initialise()
initial_status = tiger_autofocus.get_status()
calibration_success = None

if RUN_CRISP_CALIBRATION:
    calibration_success = tiger_autofocus.initialise_autofocus(
        lock_after_initialise=LOCK_AFTER_CRISP_CALIBRATION,
    )

{
    "name": tiger_autofocus.name,
    "is_initialised": tiger_autofocus.is_initialised(),
    "is_alive": tiger_autofocus.is_alive(),
    "initial_status": initial_status,
    "current_status": tiger_autofocus.get_status(),
    "is_locked": tiger_autofocus.is_locked(),
    "ran_crisp_calibration": RUN_CRISP_CALIBRATION,
    "calibration_success": calibration_success,
}


## SyncBoard

In [ ]:
# SyncBoardPeripheralController

syncboard_controller = remember_controller(
    "syncboard_controller",
    SyncBoardPeripheralController.from_serial_port(port=SYNCBOARD_PORT),
)
syncboard_controller.initialise()

{
    "name": syncboard_controller.name,
    "port": SYNCBOARD_PORT,
    "is_initialised": syncboard_controller.is_initialised(),
    "is_alive": syncboard_controller.is_alive(),
}


In [ ]:
# SyncBoardLedSource

syncboard_controller = require_object("syncboard_controller")
syncboard_led_source = remember_peripheral(
    "syncboard_led_source",
    SyncBoardLedSource(
        peripheral_ctrl=syncboard_controller,
        available_leds=[SYNCBOARD_TEST_LED],
    ),
)
syncboard_led_source.initialise()
try:
    syncboard_led_source.set_led(
        led_type=SYNCBOARD_TEST_LED,
        brightness=LED_TEST_BRIGHTNESS,
        duration=LED_TEST_DURATION_MS,
    )
    time.sleep(LED_TEST_DURATION_MS / 1000.0 + 0.05)
finally:
    syncboard_led_source.disable_led()

{
    "name": syncboard_led_source.name,
    "is_initialised": syncboard_led_source.is_initialised(),
    "is_alive": syncboard_led_source.is_alive(),
    "available_leds": syncboard_led_source.get_available_leds(),
    "test_led": SYNCBOARD_TEST_LED,
    "brightness": LED_TEST_BRIGHTNESS,
    "duration_ms": LED_TEST_DURATION_MS,
    "state_after_disable": syncboard_led_source.get_led_state(SYNCBOARD_TEST_LED),
}


In [ ]:
# SyncBoardPhotodiode

syncboard_controller = require_object("syncboard_controller")
syncboard_photodiode = remember_peripheral(
    "syncboard_photodiode",
    SyncBoardPhotodiode(
        peripheral_ctrl=syncboard_controller,
        channel=PHOTODIODE_CHANNEL,
        reading_range=PHOTODIODE_READING_RANGE,
    ),
)
syncboard_photodiode.initialise()
reading = syncboard_photodiode.read_photodiode()

{
    "name": syncboard_photodiode.name,
    "is_initialised": syncboard_photodiode.is_initialised(),
    "is_alive": syncboard_photodiode.is_alive(),
    "channel": syncboard_photodiode.channel,
    "reading_percent": reading,
}


## Cleanup

In [ ]:
# Stop and release any peripherals/controllers created above.

cleanup_errors = []

for peripheral in reversed(created_peripherals):
    with suppress(Exception):
        peripheral.stop()
    try:
        peripheral.finalise()
    except Exception as error:
        cleanup_errors.append((getattr(peripheral, "name", repr(peripheral)), repr(error)))

for controller in reversed(created_controllers):
    try:
        controller.shutdown(force=True)
    except Exception as error:
        cleanup_errors.append((getattr(controller, "name", repr(controller)), repr(error)))

{
    "cleaned_peripherals": [getattr(peripheral, "name", repr(peripheral)) for peripheral in created_peripherals],
    "cleaned_controllers": [getattr(controller, "name", repr(controller)) for controller in created_controllers],
    "cleanup_errors": cleanup_errors,
}
